# 03 · Linear & logistic regression with `nn`, and why feature scaling is not optional

Notebook 02 used bare tensors. Here you move to the real tools — `nn.Linear`, a loss object, `torch.optim` — on real-looking data, and you meet the first thing that goes wrong on real data: **features with large values make gradient descent blow up**.

---
## Theory in 60 seconds

- **Linear regression**: `ŷ = X w + b`, loss = MSE. In PyTorch: `nn.Linear(d, 1)` + `nn.MSELoss()`.
- **Logistic regression**: `p = σ(X w + b)`, loss = binary cross-entropy. In PyTorch: `nn.Linear(d, 1)` + `nn.BCEWithLogitsLoss()` (the sigmoid lives inside the loss).
- Both are a "network" with a single layer and no hidden units — the training loop is *identical* to that of an MLP.
- **Standardisation**: `x' = (x − mean) / std`, per column. Mean and std are computed on the **training** data and re-used unchanged for validation/test data.
- From 02: gradient of a weight = error × input. Large inputs → large gradients → the stable learning rate becomes tiny, and then the *other* parameters (e.g. the bias) barely move.

In [ ]:
import sys; sys.path.insert(0, "..")
import csv
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from dnd_utils import check, DATA

# One feature and the target of houses.csv (the full parsing is the topic of notebook 04).
with open(DATA / "houses.csv", newline="") as f:
    rows = list(csv.DictReader(f))
area = np.array([float(r["area_m2"]) for r in rows], dtype=np.float32)        # square metres, roughly 25 .. 220
price = np.array([float(r["price_keur"]) for r in rows], dtype=np.float32)    # thousand EUR
print(area.shape, area.mean(), area.std(), "|", price.mean(), price.std())
plt.figure(figsize=(5, 3)); plt.scatter(area, price, s=4, alpha=.4); plt.xlabel("area (m²)"); plt.ylabel("price (k€)"); plt.show()

---
## Exercise 3.1 · The training function

Write `fit(model, X, Y, lr, steps)`: full-batch gradient descent with `nn.MSELoss` and `torch.optim.SGD`. It returns the list of losses (Python floats) measured **before** each update.

Then create the tensors `X` and `Y` from `area` and `price` with the right shapes and dtype.

In [ ]:
def fit(model, X, Y, lr, steps):
    raise NotImplementedError

X = None      # (1200, 1) float32
Y = None      # (1200, 1) float32

In [ ]:
check("X shape/dtype", lambda: X.shape == (1200, 1) and X.dtype == torch.float32)
check("Y shape/dtype", lambda: Y.shape == (1200, 1) and Y.dtype == torch.float32)
torch.manual_seed(0)
check("fit returns `steps` floats", lambda: len(fit(nn.Linear(1, 1), X / 100, Y / 100, 0.1, 7)) == 7)
check("fit decreases the loss", lambda: (lambda l: l[-1] < l[0])(fit(nn.Linear(1, 1), X / 100, Y / 100, 0.1, 50)))

---
## Exercise 3.2 · Watch it explode

Run the cell. The learning rate `0.1` was perfectly fine in notebook 02.

In [ ]:
torch.manual_seed(0)
losses = fit(nn.Linear(1, 1), X, Y, lr=0.1, steps=12)
print([f"{l:.3g}" for l in losses])

**a)** Explain in one or two sentences why the loss explodes here but did not in notebook 02 (same model, same loss, same `lr`).

✍️ **Your answer** (double-click, write it in your own words *before* opening the solution):

...

**b)** For one feature and no bias, each step multiplies the error in `w` by `(1 − lr · 2·mean(x²))` (the same argument as in exercise 2.4, with curvature `2·mean(x²)` instead of `2`). Compute the largest learning rate for which the absolute value of this factor stays below 1.

In [ ]:
lr_max_theory = None

In [ ]:
print("theoretical limit:", lr_max_theory)
check("lr_max_theory", lambda: abs(lr_max_theory - 1 / float((area.astype(np.float64) ** 2).mean())) < 1e-6)
torch.manual_seed(0); check("just below the limit: still finite after 200 steps", lambda: np.isfinite(fit(nn.Linear(1, 1), X, Y, 0.95 * lr_max_theory, 200)[-1]))
torch.manual_seed(0); check("just above the limit: diverges", lambda: not np.isfinite(fit(nn.Linear(1, 1), X, Y, 1.1 * lr_max_theory, 3000)[-1]) or fit(nn.Linear(1, 1), X, Y, 1.1 * lr_max_theory, 3000)[-1] > 1e6)

**c)** So use a small learning rate? Run this and look at the **bias**. The best straight line has a bias of about `+22`.

In [ ]:
torch.manual_seed(0)
slow = nn.Linear(1, 1)
losses = fit(slow, X, Y, lr=1e-4, steps=2000)
print(f"loss after 2000 steps: {losses[-1]:.1f}   weight: {slow.weight.item():.3f}   bias: {slow.bias.item():.3f}")

The weight converged quickly; the bias has hardly moved from its initial value. Why? (Think about the curvature in the `b` direction, and the factor from 2.4.)

✍️ **Your answer** (double-click, write it in your own words *before* opening the solution):

...

---
## Exercise 3.3 · Standardise, then train

Standardise the feature (population std, i.e. NumPy's default), keep `mean_x` and `std_x`, build `Xs`, and train a fresh `nn.Linear(1, 1)` with `lr = 0.1` for 100 steps.

In [ ]:
mean_x = None
std_x = None
Xs = None                    # standardised feature, (1200, 1) float32

torch.manual_seed(0)
line = nn.Linear(1, 1)
losses_scaled = None         # train `line` on (Xs, Y)

In [ ]:
print(f"final loss {losses_scaled[-1]:.1f}   weight {line.weight.item():.2f}   bias {line.bias.item():.2f}")
check("Xs has mean 0 and std 1", lambda: abs(Xs.mean().item()) < 1e-4 and abs(Xs.std(correction=0).item() - 1) < 1e-4)
A_ = np.stack([area, np.ones_like(area)], axis=1).astype(np.float64)
best_ = ((A_ @ np.linalg.lstsq(A_, price.astype(np.float64), rcond=None)[0] - price) ** 2).mean()
check(f"reached the best possible straight line (MSE {best_:.1f})", lambda: abs(losses_scaled[-1] - best_) < 1.0)
check("bias equals the mean price (why?)", lambda: abs(line.bias.item() - price.mean()) < 0.5)

After standardisation the learned **bias equals the mean price**. Why is that — and what does the weight (≈ 63) mean in words?

✍️ **Your answer** (double-click, write it in your own words *before* opening the solution):

...

---
## Exercise 3.4 · Back to the original units

The model was trained on `x' = (x − mean_x) / std_x`:  `ŷ = w'·x' + b'`. Rewrite this as `ŷ = w·x + b` in the original units (k€ per m² and k€). Derive `w` and `b` on paper.

In [ ]:
w_orig = None      # k€ per square metre
b_orig = None      # k€

In [ ]:
ref_w, ref_b = np.polyfit(area.astype(np.float64), price.astype(np.float64), 1)
print(f"yours: {w_orig}, {b_orig}    closed form: {ref_w:.3f}, {ref_b:.3f}")
check("w in original units", lambda: abs(w_orig - ref_w) < 0.02)
check("b in original units", lambda: abs(b_orig - ref_b) < 1.5)

The same rule applies to new data: a house of 100 m² must go through **the same** `mean_x` and `std_x` before it is given to the model. Predict its price with `line`.

In [ ]:
price_100 = None      # a Python float

In [ ]:
check("prediction for 100 m²", lambda: abs(price_100 - (ref_w * 100 + ref_b)) < 1.5)

---
## Exercise 3.5 · Is the model any good? Always compare with a baseline

A loss value alone means nothing. The simplest possible "model" predicts the **mean of the training targets** for everyone.

Compute `baseline_mse`, the model's `model_mse`, and `r2 = 1 − model_mse / baseline_mse`.

In [ ]:
baseline_mse = None
model_mse = None
r2 = None

In [ ]:
print(f"baseline {baseline_mse:.0f}   model {model_mse:.0f}   R² {r2:.3f}   (RMSE: {baseline_mse ** .5:.0f} k€ -> {model_mse ** .5:.0f} k€)")
check("baseline MSE is the variance of the target", lambda: abs(baseline_mse - price.var()) < 1)
check("r2", lambda: abs(r2 - (1 - model_mse / price.var())) < 1e-3 and 0.3 < r2 < 0.6)

Area alone explains roughly half of the variance. In notebook 06 an MLP with all features will do much better. Remember the pattern: **baseline → linear model → network.** If your network is not better than the baseline, something is broken.

---
## Exercise 3.6 · Logistic regression by hand (NumPy)

Two clouds of points in 2-D; class 1 is centred at (1, 1), class 0 at (−1, −1).

Implement, vectorised:

- `sigmoid(z)`
- `bce(p, y)` — mean binary cross-entropy, with `1e-12` added inside each logarithm
- `train_logreg_numpy(X, y, lr, steps)` — weights and bias start at **zero**; each step computes `p`, records the loss (before the update), then updates with the gradients from exercise 2.3 averaged over the samples: `∂L/∂w = Xᵀ(p − y)/n`, `∂L/∂b = mean(p − y)`. Returns `(w, b, losses)`.

In [ ]:
rng = np.random.default_rng(3)
n = 400
y2 = (rng.random(n) < 0.5).astype(np.float64)                       # (400,)
X2 = rng.normal(0, 1.2, size=(n, 2)) + np.where(y2[:, None] == 1, 1.0, -1.0)   # (400, 2)
plt.figure(figsize=(4, 4)); plt.scatter(X2[:, 0], X2[:, 1], c=y2, s=6, cmap="coolwarm"); plt.show()

In [ ]:
def sigmoid(z):
    raise NotImplementedError

def bce(p, y):
    raise NotImplementedError

def train_logreg_numpy(X, y, lr, steps):
    raise NotImplementedError

In [ ]:
check("sigmoid(0) = 0.5, vectorised", lambda: np.allclose(sigmoid(np.array([0.0, 100.0, -100.0])), [0.5, 1.0, 0.0]))
check("bce of a coin-flip prediction is ln 2", lambda: abs(bce(np.full(4, 0.5), np.array([0., 1., 1., 0.])) - np.log(2)) < 1e-9)
w_np, b_np, losses_np = train_logreg_numpy(X2, y2, lr=0.5, steps=200)
check("first loss is ln 2 ≈ 0.693", lambda: abs(losses_np[0] - np.log(2)) < 1e-9)
check("loss decreases", lambda: losses_np[-1] < 0.35)
check("both weights are positive (class 1 is up-right)", lambda: (w_np > 0.5).all())

Why is the very first loss exactly `ln 2 ≈ 0.693`, regardless of the data?

✍️ **Your answer** (double-click, write it in your own words *before* opening the solution):

...

---
## Exercise 3.7 · The same thing in PyTorch

Reproduce the NumPy run with `nn.Linear(2, 1)` (weights and bias set to zero), `nn.BCEWithLogitsLoss()` and `torch.optim.SGD(lr=0.5)`, 200 full-batch steps. Record the losses before each update in `losses_torch`.

If both implementations are right, the two loss curves coincide to 4 decimals.

<details><summary>Hint — zero initialisation</summary>

```python
with torch.no_grad():
    model.weight.zero_()
    model.bias.zero_()
```
</details>

In [ ]:
Xt = None        # (400, 2) float32
yt = None        # (400, 1) float32   <- shape of the model output!

logreg = nn.Linear(2, 1)
losses_torch = []

In [ ]:
check("same loss curve as NumPy", lambda: len(losses_torch) == 200 and np.allclose(losses_torch, losses_np, atol=1e-4))
check("same weights as NumPy", lambda: np.allclose(logreg.weight.detach().numpy().reshape(-1), w_np, atol=1e-3))

---
## Exercise 3.8 · From logits to decisions

The model outputs a **logit** `z`. Compute, for the training points:

- `probs`: predicted probabilities of class 1, shape `(400, 1)`
- `pred`: predicted class (0.0 or 1.0) using threshold 0.5, shape `(400, 1)`
- `acc`: accuracy as a Python float
- `pred_from_logits`: the same predictions **without computing the sigmoid** (which condition on `z` is equivalent to `p > 0.5`?)

In [ ]:
probs = None
pred = None
acc = None
pred_from_logits = None

In [ ]:
check("probs in (0, 1) with shape (400, 1)", lambda: probs.shape == (400, 1) and 0 < probs.min() and probs.max() < 1)
check("accuracy is clearly better than chance", lambda: 0.85 < acc < 1.0)
check("threshold on the logit gives identical predictions", lambda: (pred_from_logits == pred).all().item())

The decision boundary is where `z = w₁x₁ + w₂x₂ + b = 0` — a straight line. A logistic regression can only ever separate classes with a line (a hyperplane). To bend the boundary you need hidden layers with a non-linearity: the MLP of notebook 06.

---
## Blank-page drill

From memory, in the empty cell: given `area` and `price` (NumPy, shape `(1200,)`),

1. standardise `area`,
2. build `X`, `Y` with the right shapes and dtype,
3. train `nn.Linear(1, 1)` with `MSELoss` and `SGD(lr=0.1)` for 100 full-batch steps,
4. print the final loss and the RMSE in k€.

You should land at an MSE of roughly 4 000 (RMSE ≈ 63 k€).

In [ ]:
# your code